In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ==========================================
# 1. DATASET: Points (X,y) -> Target Vector
# ==========================================
def get_valid_indices(npz_path, total_expected=50000):
    data = np.load(npz_path)
    valid_indices = []

    print("Scanning for valid data points...")
    for i in range(total_expected):
        try:
            # Check if index exists AND has data
            if f"X_{i}" in data and data[f"X_{i}"].shape[0] > 0:
                valid_indices.append(i)
        except:
            continue

    print(f"Found {len(valid_indices)} valid equations out of {total_expected}.")
    return valid_indices

# --- Update your Dataset Class to use this list ---
class JEPADataset(Dataset):
    def __init__(self, npz_path, npy_path, valid_indices):
        print("Loading data into RAM for maximum speed...")
        full_data = np.load(npz_path)
        all_targets = np.load(npy_path)

        # Pre-allocate numpy arrays in RAM to avoid file I/O during training
        self.points = np.zeros((len(valid_indices), 500, 4), dtype=np.float32)
        self.targets = np.zeros((len(valid_indices), 128), dtype=np.float32)

        for i, real_idx in enumerate(tqdm(valid_indices, desc="Pre-loading")):
            x_pts = full_data[f"X_{real_idx}"]
            y_pts = full_data[f"y_{real_idx}"]

            # Efficient slicing
            n_p = min(x_pts.shape[0], 500)
            n_v = min(x_pts.shape[1], 3)

            self.points[i, :n_p, :n_v] = x_pts[:n_p, :n_v]
            self.points[i, :n_p, 3] = y_pts[:n_p]
            self.targets[i] = all_targets[real_idx]

        # Convert to tensors once
        self.points = torch.from_numpy(self.points)
        self.targets = torch.from_numpy(self.targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.points[idx], self.targets[idx]# ==========================================
# 2. ARCHITECTURE: Context Encoder & Predictor
# ==========================================
class TransformerContextEncoder(nn.Module):
    def __init__(self, input_dim=4, embed_dim=128, nhead=4, num_layers=2):
        super().__init__()
        # 1. Project 4D points (x1,x2,x3,y) to a higher dimensional space
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU()
        )

        # 2. Transformer Encoder Layers
        # This allows points to "talk" to each other to find patterns
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * 2,
            batch_first=True,
            dropout=0.05
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Final Prediction Head (The Predictor)
        self.predictor = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )

    def forward(self, x):
        # x shape: (Batch, 500, 4)

        # Project to embedding space
        x = self.input_proj(x) # (Batch, 500, 128)

        # Pass through Transformer
        # We don't need positional encoding because the order of points doesn't matter
        x = self.transformer(x) # (Batch, 500, 128)

        # Global Pooling (Max + Mean) to capture both average and extreme features
        mean_pool = x.mean(dim=1)
        max_pool, _ = x.max(dim=1)
        pooled = mean_pool + max_pool # (Batch, 128)

        # Predict the Phase 1 target embedding
        return self.predictor(pooled)

class Predictor(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.ReLU(),
            nn.Linear(dim * 2, dim) # Maps context to target space
        )

    def forward(self, z):
        return self.net(z)




In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset
import torch

class DecoderWithEmbeddingDataset(Dataset):
    def __init__(self, embeddings, strings):
        self.embeddings = torch.tensor(embeddings).float()

        self.strings = strings

    def __len__(self):
        return len(self.strings)

    def __getitem__(self, idx):
        z = self.embeddings[idx]

        tokens = [TOKEN2ID["<BOS>"]] + \
                 [TOKEN2ID.get(t, TOKEN2ID["<UNK>"]) for t in self.strings[idx].split()] + \
                 [TOKEN2ID["<EOS>"]]

        if len(tokens) < MAX_LEN:
            tokens += [PAD_ID] * (MAX_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_LEN]

        tokens = torch.tensor(tokens)
        input_ids = tokens[:-1].clone()
        labels = tokens[1:].clone()
        labels[labels == PAD_ID] = -100


        return z, input_ids, labels

class JEPADecoder(nn.Module):
  def __init__(self, latent_dim=128, hidden_dim=768, vocab_size=27, max_len=30):
      super().__init__()
      self.max_len = max_len
      self.vocab_size = vocab_size

      # Initial projection: Latent Vector -> RNN Hidden State
      self.latent_to_h0 = nn.Linear(latent_dim, hidden_dim)

      self.embedding = nn.Embedding(vocab_size, hidden_dim)
      self.gru = nn.GRU(hidden_dim, hidden_dim,num_layers=2 ,batch_first=True)
      self.fc_out = nn.Linear(hidden_dim, vocab_size)

  def forward(self, z, target_tokens=None, teacher_forcing_ratio=0.5):
      batch_size = z.size(0)
      h0 = self.latent_to_h0(z)                # (B, hidden_dim)
      h0 = h0.unsqueeze(0)                    # (1, B, hidden_dim)
      h0 = h0.repeat(self.gru.num_layers, 1, 1)  # (num_layers, B, hidden_dim)

      hidden = h0
      # Initialize hidden state from JEPA latent vector
      # hidden = self.latent_to_h0(z).unsqueeze(0) # (1, B, Hidden)

      # Start token
      input_id = torch.full((batch_size, 1), TOKEN2ID["<BOS>"], device=z.device)
      all_logits = []
      seq_len = target_tokens.size(1) if target_tokens is not None else self.max_len

      for t in range(seq_len):


        embedded = self.embedding(input_id) # (B, 1, Hidden)
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc_out(output) # (B, 1, Vocab)
        all_logits.append(logits)

        # Decide next input
        if target_tokens is not None and torch.rand(1).item() < teacher_forcing_ratio:
            input_id = target_tokens[:, t].unsqueeze(1) # Use Ground Truth
        else:
            input_id = logits.argmax(dim=-1) # Use Model Prediction

        if (not self.training) and (input_id == TOKEN2ID["<EOS>"]).all():
            break

      return torch.cat(all_logits, dim=1)

In [28]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

def convert_folder_to_npz(master_csv_path, data_folder, save_path, max_points=500):

    master_df = pd.read_csv(master_csv_path)
    npz_dict = {}

    print("Building NPZ from space-separated files...")

    for idx, row in tqdm(master_df.iterrows(), total=len(master_df)):

        filename = row["filename"]
        file_path = os.path.join(data_folder, filename)

        # Read space-separated, no header
        df = pd.read_csv(file_path, sep="\s+", header=None)
        # X = all columns except last
        X = df.iloc[:, :-1].values.astype(np.float32)

        # y = last column
        y = df.iloc[:, -1].values.astype(np.float32)

        # Optional: Subsample to 500 points
        if len(X) > max_points:
            indices = np.random.choice(len(X), max_points, replace=False)
            X = X[indices]
            y = y[indices]

        npz_dict[f"X_{idx}"] = X
        npz_dict[f"y_{idx}"] = y

    np.savez_compressed(save_path, **npz_dict)

    print(f"Saved structured NPZ at: {save_path}")

<>:19: SyntaxWarning: invalid escape sequence '\s'
<>:19: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_457/109175065.py:19: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(file_path, sep="\s+", header=None)


In [8]:
convert_folder_to_npz("/content/drive/MyDrive/Colab Notebooks/symbolic_data/feynman_new.csv", data_folder="/content/drive/MyDrive/Colab Notebooks/symbolic_data/Feynman_with_units/Feynman_with_units",save_path = "/content/drive/MyDrive/Colab Notebooks/symbolic_data/feynman_datapoints.npz")

Building NPZ from space-separated files...


100%|██████████| 100/100 [03:09<00:00,  1.89s/it]


Saved structured NPZ at: /content/drive/MyDrive/Colab Notebooks/symbolic_data/feynman_datapoints.npz


In [4]:
import torch
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/tiny_target_encoder_feynman_21pct (1).pt", map_location=device)
TOKEN2ID = checkpoint['vocab']
# Load structured NPZ
npz_data = np.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz")

# Load equation strings
master_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/symbolic_data/equations_50k.csv")
equations = master_df["expression_prefix_masked"].tolist()

# Load encoder
encoder = TransformerContextEncoder()
encoder.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/transformer_context_encoder.pt", map_location=device))
encoder.to(device).eval()

# Load decoder
decoder = JEPADecoder(
    latent_dim=128,
    hidden_dim=768,
    vocab_size=len(TOKEN2ID),
    max_len=31
)
decoder.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/jepa_decoder_3_march_updraded.pt", map_location=device))
decoder.to(device).eval()

JEPADecoder(
  (latent_to_h0): Linear(in_features=128, out_features=768, bias=True)
  (embedding): Embedding(27, 768)
  (gru): GRU(768, 768, num_layers=2, batch_first=True)
  (fc_out): Linear(in_features=768, out_features=27, bias=True)
)

In [5]:
ID2TOKEN = {v: k for k, v in TOKEN2ID.items()}

def ids_to_string(ids):
    tokens = []
    for i in ids:
        token = ID2TOKEN.get(i, "<UNK>")
        if token in ["<BOS>", "<PAD>"]:
            continue
        if token == "<EOS>":
            break
        tokens.append(token)
    return " ".join(tokens)

In [6]:
@torch.no_grad()
def generate_from_points(points):
    """
    points: (1, 500, 4)
    """

    z = encoder(points.to(device))
    logits = decoder(z, target_tokens=None)

    preds = logits.argmax(dim=-1)
    return ids_to_string(preds.squeeze(0).cpu().tolist())

In [10]:
import numpy as np
import torch

def uses_only_3_vars(equation_string):
    # Check for x4, x5, x6, etc. up to x9
    for i in range(4, 10):
        if f'x{i}' in equation_string:
            return False
    return True

def evaluate_model(npz_path, equations):

    # Load data once
    npz_data = np.load(npz_path)

    # Get valid indices
    valid_indices = get_valid_indices(npz_path, total_expected=len(equations))

    exact_correct = 0
    token_correct = 0
    token_total = 0
    total = 0

    print("Evaluating model...")

    for i in valid_indices:

        gt_eq = equations[i]

        # Skip equations using more than 3 variables
        if not uses_only_3_vars(gt_eq):
            continue

        try:
            X = npz_data[f"X_{i}"]
            y = npz_data[f"y_{i}"]

            pts = np.zeros((500, 4), dtype=np.float32)

            n_p = min(len(X), 500)

            n_vars = X.shape[1]
            n_copy = min(n_vars, 3)

            pts[:n_p, :n_copy] = X[:n_p, :n_copy]
            pts[:n_p, 3] = y[:n_p]

            pts_tensor = torch.tensor(pts).unsqueeze(0).float()

            pred_eq = generate_from_points(pts_tensor)

        except Exception:
            continue

        total += 1

        # Exact match
        if pred_eq.strip() == gt_eq.strip():
            exact_correct += 1

        # Token accuracy
        gt_tokens = gt_eq.split()
        pred_tokens = pred_eq.split()

        min_len = min(len(gt_tokens), len(pred_tokens))

        for t in range(min_len):
            if gt_tokens[t] == pred_tokens[t]:
                token_correct += 1

        token_total += min_len

    print("\n========== RESULTS ==========")
    print(f"Evaluated on {total} valid equations")

    if total > 0:
        print(f"Exact Match Accuracy: {exact_correct / total:.4f}")

    if token_total > 0:
        print(f"Token Accuracy: {token_correct / token_total:.4f}")

In [11]:
npz_file_path = "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_last_2_march/equations_50k_data.npz"
evaluate_model(npz_file_path, equations)

Scanning for valid data points...
Found 44820 valid equations out of 50000.
Evaluating model...

========== RESULTS ==========
Evaluated on 44820 valid equations
Exact Match Accuracy: 0.0000
Token Accuracy: 0.1057
